# dispersion_meta — End-to-End Walkthrough

This notebook demonstrates the full daily workflow:

1. **Propose** — the bandit selects optimizer configs and produces proposals
2. **Record outcomes** — after T+5, compute forward realized stats
3. **Record decisions** — log accept/reject for each proposal
4. **Inspect** — read back the stored data

We use synthetic data throughout so no external market data is needed.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import polars as pl
from datetime import date
from pathlib import Path

from dispersion_meta import io, schemas
from dispersion_meta.paths import set_data_root, data_root
from dispersion_meta.config_space import CONFIG_GRID, FAMILIES, grid_size, total_grid_size
from dispersion_meta.bandit import BayesianLinearBandit
from dispersion_meta.synthetic import FEATURE_NAMES, MetaScoreOracle

## 0. Setup — use a temp directory so we don't pollute real data

In [ ]:
import tempfile

tmp_dir = Path(tempfile.mkdtemp(prefix="dispersion_meta_demo_"))
set_data_root(tmp_dir)
print(f"Data root: {data_root()}")

## 1. Explore the config space

The bandit selects among 5 objective families. Each family has a discrete grid of optimizer configs.

In [ ]:
print(f"Families: {FAMILIES}")
print(f"Total configs: {total_grid_size()}")
print()
for fam in FAMILIES:
    print(f"  {fam}: {grid_size(fam)} configs")

In [ ]:
# Example config from each family
for fam in FAMILIES:
    print(f"\n--- {fam} (first config) ---")
    for k, v in CONFIG_GRID[fam][0].items():
        print(f"  {k}: {v}")

## 2. Run proposals (Day 1 — cold start)

On day 1 there's no training data, so the bandit samples from its prior (approximately uniform family selection).

We mock the optimizer with a simple fake since this is a demo. In production you'd have `dispersion_optimization` installed and pass real PnL matrices.

In [ ]:
from unittest.mock import patch
from dataclasses import dataclass

@dataclass
class FakeResult:
    weights: np.ndarray
    n_names: int
    selected_names: list
    sharpe: float = 1.5
    mean_pnl: float = 0.01
    max_drawdown: float = -0.03
    status: str = "optimal"
    solve_time: float = 0.05

def fake_optimizer(config, pnl_matrix, column_names):
    n = pnl_matrix.shape[1]
    w = np.ones(n) / n
    return FakeResult(weights=w, n_names=n, selected_names=list(range(n)))

In [ ]:
from dispersion_meta.propose import propose_today

# Simulated inputs
products = ["corridor_var", "vol", "gamma", "dngamma", "var"]
rng = np.random.default_rng(42)
n_names = 30

features = {
    "vix_level": 18.5,
    "vix_20d_change": -0.02,
    "avg_pairwise_corr": 0.42,
    "dispersion_pnl_20d": 0.013,
    "skew_steepness": 0.08,
    "term_structure_slope": 0.015,
    "earnings_density_21d": 0.07,
}

pnl_matrices = {p: rng.normal(0.0, 0.01, (60, n_names)) for p in products}
column_names = [f"S{i:02d}" for i in range(n_names)]

with patch("dispersion_meta.propose._run_optimizer", side_effect=fake_optimizer):
    result = propose_today(
        today=date(2026, 4, 16),
        features=features,
        pnl_matrices=pnl_matrices,
        column_names=column_names,
        seed=42,
    )

print(f"Total proposals: {result['n_proposals_total']}")
print(f"Products: {result['n_products']}")
for product, info in result["products"].items():
    print(f"  {product}: {info['n_proposals']} proposals — families: {info['families']}")

## 3. Inspect stored proposals

In [ ]:
proposals = io.read_proposals()
print(f"Stored proposals: {len(proposals)} rows")
proposals.select(
    "date", "product", "family", "proposal_type",
    "thompson_sample_value", "posterior_mean", "posterior_std",
    "solver_status", "n_names",
)

## 4. Record outcomes (T+5)

After 5 trading days, compute forward realized stats and meta-scores.

In [ ]:
from dispersion_meta.record_outcome import record_outcomes

forward_returns = {p: rng.normal(0.001, 0.01, (5, n_names)) for p in products}
trailing_vol = {p: round(rng.uniform(0.01, 0.02), 4) for p in products}

outcomes = record_outcomes(
    propose_date=date(2026, 4, 16),
    eval_date=date(2026, 4, 23),
    forward_returns=forward_returns,
    trailing_vol=trailing_vol,
    column_names=column_names,
)

print(f"Outcomes written: {len(outcomes)} rows")
outcomes.select(
    "date", "product", "config_hash", "eval_date",
    "forward_5d_pnl", "forward_sharpe", "meta_score", "meta_score_mode",
)

## 5. Record a decision

In [ ]:
from dispersion_meta.record_decision import record_decision

# Accept the 'best' proposal for corridor_var
best_proposal = proposals.filter(
    (pl.col("product") == "corridor_var") & (pl.col("proposal_type") == "best")
)
best_hash = best_proposal["config_hash"][0]

record_decision(
    propose_date=date(2026, 4, 16),
    config_hash=best_hash,
    product="corridor_var",
    decision="accepted",
    notes="Looks reasonable for current regime",
)

decisions = io.read_decisions_raw()
print(f"Decisions recorded: {len(decisions)}")
decisions.select("date", "product", "config_hash", "decision", "notes")

## 6. Training table — what the bandit sees

The training table joins proposals, features, and outcomes with walk-forward filtering.

In [ ]:
from dispersion_meta.training_table import build_training_table

training = build_training_table(as_of=date(2026, 4, 30))
if training is not None:
    print(f"Training table: {training.shape}")
    print(f"Columns: {training.columns}")
    training.select(
        "date", "product", "family", "meta_score",
        *FEATURE_NAMES,
    ).head(10)
else:
    print("No training data yet (need proposals + outcomes)")

## 7. Bandit internals — fitting and Thompson sampling

Here we use synthetic oracle data to show the bandit learning which families work best in different regimes.

In [ ]:
from dispersion_meta.synthetic import generate_features, synth_proposals_for_day, synth_outcome_rows
from datetime import timedelta

# Generate 200 days of synthetic data
oracle = MetaScoreOracle()
synth_rng = np.random.default_rng(42)
n_days = 200
start = date(2024, 1, 2)
synth_products = ["corridor_var"]

feat_df = generate_features(n_days, synth_products, start_date=start, rng=synth_rng)
all_props = []
for i in range(n_days):
    dt = start + timedelta(days=i)
    day_props = synth_proposals_for_day(dt, synth_products, n_per_family=2, n_names=20, rng=synth_rng)
    all_props.append(day_props)
props_df = pl.concat(all_props)
outcomes_df = synth_outcome_rows(props_df, feat_df, oracle, rng=synth_rng)

# Join into training table format
training_synth = props_df.join(feat_df, on=["date", "product"], how="inner", suffix="_feat")
training_synth = training_synth.join(outcomes_df, on=["date", "product", "config_hash"], how="inner", suffix="_out")

print(f"Synthetic training table: {training_synth.shape}")

In [ ]:
# Fit the bandit
bandit = BayesianLinearBandit()
bandit.fit(training_synth)

# Show diagnostics
diag = bandit.diagnostics()
print("Per-arm diagnostics:")
for fam, info in diag["arms"].items():
    print(f"  {fam:15s}  n_obs={info['n_obs']:4d}  "
          f"posterior_mean_norm={info['posterior_mean_norm']:.3f}  "
          f"posterior_trace={info['posterior_trace']:.4f}")

In [ ]:
# Thompson sample under two regimes
x_high_vix = np.array([35.0, 5.0, 0.2, 0.0, 0.0, 0.0, 0.1])
x_low_vix = np.array([12.0, -2.0, 0.6, 0.5, 0.0, 0.0, 0.1])

ts_rng = np.random.default_rng(0)

print("High VIX regime — posterior means per family:")
for s in bandit.thompson_sample(x_high_vix, ts_rng):
    print(f"  {s['family']:15s}  posterior_mean={s['posterior_mean']:+.3f}  "
          f"posterior_std={s['posterior_std']:.3f}")

print("\nLow VIX / high corr regime — posterior means per family:")
for s in bandit.thompson_sample(x_low_vix, ts_rng):
    print(f"  {s['family']:15s}  posterior_mean={s['posterior_mean']:+.3f}  "
          f"posterior_std={s['posterior_std']:.3f}")

In [ ]:
# Select proposals for the high-VIX regime
proposals_selected = bandit.select_proposals(x_high_vix, np.random.default_rng(0))

print("Selected proposals (high VIX):")
for p in proposals_selected:
    print(f"  {p['proposal_type']:8s}  family={p['family']:15s}  "
          f"sampled={p['thompson_sample_value']:+.3f}  "
          f"post_mean={p['posterior_mean']:+.3f}  "
          f"post_std={p['posterior_std']:.3f}")

## 8. Cleanup

In [ ]:
import shutil
set_data_root(None)
shutil.rmtree(tmp_dir, ignore_errors=True)
print("Done — temp data cleaned up.")